In [4]:
import pandas as pd
import numpy as np


df = pd.read_csv("../data/telco_churn.csv")


df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Data loaded. Shape:", df.shape)
print("Churn column:", df['Churn'].unique())

Data loaded. Shape: (7043, 21)
Churn column: [0 1]


In [5]:
# Separate columns by type
text_columns = df.select_dtypes(include='object').columns.tolist()
number_columns = df.select_dtypes(include=['int64','float64']).columns.tolist()

print("TEXT columns (need conversion):")
for col in text_columns:
    print(f"  {col}: {df[col].unique()}")

print("\nNUMBER columns (ready to use):")
print(number_columns)

TEXT columns (need conversion):
  customerID: ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
  gender: ['Female' 'Male']
  Partner: ['Yes' 'No']
  Dependents: ['No' 'Yes']
  PhoneService: ['No' 'Yes']
  MultipleLines: ['No phone service' 'No' 'Yes']
  InternetService: ['DSL' 'Fiber optic' 'No']
  OnlineSecurity: ['No' 'Yes' 'No internet service']
  OnlineBackup: ['Yes' 'No' 'No internet service']
  DeviceProtection: ['No' 'Yes' 'No internet service']
  TechSupport: ['No' 'Yes' 'No internet service']
  StreamingTV: ['No' 'Yes' 'No internet service']
  StreamingMovies: ['No' 'Yes' 'No internet service']
  Contract: ['Month-to-month' 'One year' 'Two year']
  PaperlessBilling: ['Yes' 'No']
  PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

NUMBER columns (ready to use):
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [6]:


binary_columns = ['gender', 'Partner', 'Dependents', 'PhoneService',
                  'PaperlessBilling']


binary_map = {
    'Yes': 1, 'No': 0,
    'Male': 1, 'Female': 0
}

for col in binary_columns:
    df[col] = df[col].map(binary_map)
    print(f"{col} converted: {df[col].unique()}")

gender converted: [0 1]
Partner converted: [1 0]
Dependents converted: [0 1]
PhoneService converted: [0 1]
PaperlessBilling converted: [1 0]


one-hot encoding --> create new col. for each possible value and pd.get_dummies does one-hot encoding automatically


In [7]:
multi_columns = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                 'StreamingTV', 'StreamingMovies', 'Contract',
                 'PaymentMethod']


df = pd.get_dummies(df, columns=multi_columns, drop_first=True)

print("Shape after encoding:", df.shape)
print("New columns created:")
print([col for col in df.columns if any(m in col for m in multi_columns)])

Shape after encoding: (7043, 32)
New columns created:
['MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [8]:
# Feature 1: Charges per month of tenure (value for money ratio)
# Avoid division by zero for new customers (tenure=0)
df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# Feature 2: Is this a new customer? (high risk group )
df['is_new_customer'] = (df['tenure'] <= 6).astype(int)

# Feature 3: Has multiple services? (more invested = less likely to leave)
df['total_services'] = (df['PhoneService'] + 
                        df['PaperlessBilling'] +
                        df['Partner'] +
                        df['Dependents'])

print("New features created:")
print(df[['charges_per_tenure', 
          'is_new_customer', 
          'total_services']].describe())

New features created:
       charges_per_tenure  is_new_customer  total_services
count         7043.000000      7043.000000     7043.000000
mean             5.770645         0.210280        2.278007
std              8.722435         0.407536        0.973642
min              0.264384         0.000000        0.000000
25%              1.250000         0.000000        2.000000
50%              2.075926         0.000000        2.000000
75%              5.946429         0.000000        3.000000
max             80.850000         1.000000        4.000000


In [9]:
# customerID is just a random ID number — tells model nothing
# Drop it so it doesn't confuse the model
df = df.drop('customerID', axis=1)

# Check final shape
print("Final shape:", df.shape)
print("\nAll columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

Final shape: (7043, 34)

All columns:
  1. gender
  2. SeniorCitizen
  3. Partner
  4. Dependents
  5. tenure
  6. PhoneService
  7. PaperlessBilling
  8. MonthlyCharges
  9. TotalCharges
  10. Churn
  11. MultipleLines_No phone service
  12. MultipleLines_Yes
  13. InternetService_Fiber optic
  14. InternetService_No
  15. OnlineSecurity_No internet service
  16. OnlineSecurity_Yes
  17. OnlineBackup_No internet service
  18. OnlineBackup_Yes
  19. DeviceProtection_No internet service
  20. DeviceProtection_Yes
  21. TechSupport_No internet service
  22. TechSupport_Yes
  23. StreamingTV_No internet service
  24. StreamingTV_Yes
  25. StreamingMovies_No internet service
  26. StreamingMovies_Yes
  27. Contract_One year
  28. Contract_Two year
  29. PaymentMethod_Credit card (automatic)
  30. PaymentMethod_Electronic check
  31. PaymentMethod_Mailed check
  32. charges_per_tenure
  33. is_new_customer
  34. total_services


In [10]:
# Save the fully processed dataframe
df.to_csv("../data/telco_churn_processed.csv", index=False)

print("✅ Processed data saved!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.shape[1]}")
print(f"Rows: {df.shape[0]}")

✅ Processed data saved!
Shape: (7043, 34)
Columns: 34
Rows: 7043
